# COMP3334 - Computer System Security

## Tutorial 1: Cryptography

Python has some built-in modules for secure programming. 

### Random Number Generator

#### os.urandom(size)

Python has a low-level function *os.urandom(size)* to generate *size* cryptographically-secure random bytes from Operating System's interface.

You do not need to provide a seed because your operating system manages it.

In [ ]:
import os
rand_1 = os.urandom(4) # generate 4-byte random data
print(rand_1.hex())    # output the random number in hexadecimal

#### Module *secrets*

Python provides a module, **secrets**, which leverages *urandom* to generate cryptographically-secure random data in various types. 

In [ ]:
import secrets

##### secrets.randbits(k)

It returns a non-negative integer with k random bits.

In [ ]:
rand_2 = secrets.randbits(8)    # return a 8-bit random number
print(rand_2)

##### secrets.token_bytes(nbytes)

Generate *nbytes* cryptographically-secure random bytes. 

It is equivalent to *os.urandom(size)*.

In [ ]:
rand_3 = secrets.token_bytes(4)    # generate 4-byte random data
print(rand_3.hex())

##### secrets.choice(seq)

Choose an item from *seq* randomly. 

In [ ]:
range_a = range(10)                # range_a = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
rand_4 = secrets.choice(range_a)   # Choose one item randomly
print(rand_4)

### Caesar Cipher

Function *charToNum(c)* converts a character *c* to its corresponding number.

Function *numToChar(n)* converts a number *n* to its corresponding character.

We only consider lower-case characters in this tutorial. 

In [ ]:
import string


def charToNum(c):
    charTab = list(string.ascii_lowercase)
    if c not in charTab:
        raise Exception('Invalid Character!')
    return charTab.index(c)

def numToChar(n):
    charTab = list(string.ascii_lowercase)
    if int(n) < 0 or int(n) > 25:
        raise Exception('Invalid Number!')
    return charTab[int(n)]

The following is an example. 

In [ ]:
print(charToNum('c'))
print(numToChar(24))

In [ ]:
def caesar_enc(k, m):   # k: the secret key of Caesar Cipher; m: the plaintext message
    c = ''
    for letter in m:
        letter_n = charToNum(letter)
        
        # TODO: COMPLETE THE CODE TO ENCRYPT letter_n WITH THE SECRET KEY k. YOU SHOULD STILL USE letter_n TO STORE THE ENCRYPTED NUMBER
        

        c += numToChar(letter_n)
    return c

def caesar_dec(k, c):   # k: the secret key of Caesar Cipher; c: the ciphertext
    m = ''
    for letter in c:
        letter_n = charToNum(letter)

        # TODO: COMPLETE THE CODE TO DECRYPT letter_n WITH THE SECRET KEY k. YOU SHOULD STILL USE letter_n TO STORE THE DECRYPTED NUMBER
        

        m += numToChar(letter_n)
    return m


The following code is used to test whether your program is correct. 

Your program is correct if nothing is outputted in the following block. 

In [ ]:
caesar_k = 3            # the secret key of Caesar Cipher
plaintext = 'helloworld'

ciphertext = caesar_enc(caesar_k, plaintext)
assert ciphertext == 'khoorzruog'

dec_plaintext = caesar_dec(caesar_k, ciphertext)
assert dec_plaintext == plaintext

### Hash and HMAC

Python also has built-in modules for hashing and HMAC operations. 

#### hashlib

The module, *hashlib*, helps us generate digest for a message. 

You can use the algorithm you want to use. For example:

*hashlib.md5()*

*hashlib.sha1()*

*hashlib.sha224()*

*hashlib.sha256()*

*hashlib.sha384()*

*hashlib.sha512()*

*hashlib.sha3_224()*

*hashlib.sha3_256()*

*hashlib.sha3_384()*

*hashlib.sha3_512()*

...

Refer to [this link](https://docs.python.org/3/library/hashlib.html) for details. 

In [ ]:
import hashlib

m1 = hashlib.sha256()    # create an engine to generate the digest based on SHA256

# We want to hash the message "hello world" (without quotes).
m1.update(b'hello')
m1.update(b' world')
print(m1.hexdigest())

m2 = hashlib.sha256(b'hello world')
print(m2.hexdigest())

assert m1.hexdigest() == m2.hexdigest()

#### hmac

The module, *hmac*, helps us generate HMAC for a message with a key.

In [ ]:
import hmac

# Use the SHA-256 digest of a password "password" as the secret key of HMAC.
m3 = hashlib.sha256(b'password')
hmac_key = m3.digest()

# We want to hash the message "hello world" (without quotes) by HMAC-SHA256
mac = hmac.digest(hmac_key, b"hello world", hashlib.sha256)

print(mac.hex())

### Encryption Algorithms

Make sure you have installed the package *cryptography*.

Execute *pip install cryptography* in your command line and restart Jupyter. 

#### AES

In the following block, we use AES with CBC mode as an example. 

In [ ]:
import os
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

AES_k = os.urandom(32)    # generate the 32-byte (256-bit) random data as the secret key of AES
AES_iv = os.urandom(16)   # generate the 16-byte (128-bit) random data as the initialization vector of AES
cipher = Cipher(algorithms.AES(AES_k), modes.CBC(AES_iv))
encryptor = cipher.encryptor()
ciphertext = encryptor.update(b'a secret message') + encryptor.finalize()

decryptor = cipher.decryptor()
dec_plaintext = decryptor.update(ciphertext) + decryptor.finalize()
print(dec_plaintext)
assert dec_plaintext == b'a secret message'

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa, utils, padding
from cryptography.hazmat.primitives import hashes

private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public_key = private_key.public_key()

m = b'hello world'
c = public_key.encrypt(m, padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()), algorithm=hashes.SHA256(), label=None))
dec_plaintext = private_key.decrypt(c, padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()), algorithm=hashes.SHA256(), label=None))
print(dec_plaintext)
assert dec_plaintext == m